<a href="https://colab.research.google.com/github/pantera-rosa/vuln-management-capstone/blob/eric_branch/OSV_Log4j_Extras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OSV → Log4j Focused Visualizations

This notebook fetches **OSV** vulnerability data for **Apache Log4j Core** (`Maven: org.apache.logging.log4j:log4j-core`) and generates focused EDA charts:

- Timeline of Log4j vulnerabilities (published dates, severity)
- Count by year
- Version-range view (introduced → fixed) per vulnerability
- Time-to-fix histogram (days from publish to earliest fixed version)
- CSV exports of normalized records and per-vulnerability version ranges


In [ ]:

# Imports & output folder
import textwrap, time, datetime as dt
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path("/mnt/data")
NOW_STR = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = BASE_DIR / f"osv_log4j_outputs_{NOW_STR}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_plot(fig, name):
    p = OUT_DIR / f"{name}.png"
    fig.savefig(p, bbox_inches="tight", dpi=150)
    plt.close(fig)
    return p

def _print(msg):
    print(textwrap.fill(msg, width=100))


In [ ]:

# Fetch OSV data for Log4j (Maven)
OSV_QUERY_URL = "https://api.osv.dev/v1/query"

def api_query_package(ecosystem: str, name: str, version: str = None):
    payload = {"package": {"ecosystem": ecosystem, "name": name}}
    if version:
        payload["version"] = version
    r = requests.post(OSV_QUERY_URL, json=payload, timeout=60)
    r.raise_for_status()
    return r.json().get("vulns", []) or []

ECO = "Maven"
PKG = "org.apache.logging.log4j:log4j-core"

raw = api_query_package(ECO, PKG)
_print(f"Fetched {len(raw)} OSV records for {ECO}/{PKG}")


Fetched 7 OSV records for Maven/org.apache.logging.log4j:log4j-core


In [ ]:

# Normalize and flatten OSV records for plotting
def extract_severity_max(osv):
    sev = osv.get("severity") or []
    scores = []
    for s in sev:
        sc = s.get("score")
        if sc is None:
            continue
        try:
            scores.append(float(sc))
        except:
            pass
    return max(scores) if scores else None

def get_alias_cve(osv):
    aliases = osv.get("aliases") or []
    for a in aliases:
        if a.startswith("CVE-"):
            return a
    return None

def flatten_ranges(affected):
    rows = []
    for a in affected or []:
        pkg = (a.get("package") or {})
        if pkg.get("ecosystem") != "Maven" or pkg.get("name") != "org.apache.logging.log4j:log4j-core":
            continue
        versions = a.get("versions") or []
        events = []
        for r in a.get("ranges") or []:
            for ev in r.get("events", []):
                events.append({
                    "introduced": ev.get("introduced"),
                    "fixed": ev.get("fixed"),
                    "last_affected": ev.get("last_affected"),
                })
        rows.append({"versions": versions, "events": events})
    return rows

norm_rows = []
range_rows = []  # per-vuln ranges table
for rec in raw:
    row = {
        "id": rec.get("id"),
        "cve": get_alias_cve(rec),
        "published": rec.get("published"),
        "modified": rec.get("modified"),
        "summary": rec.get("summary"),
        "severity_max": extract_severity_max(rec),
    }
    norm_rows.append(row)
    fr = flatten_ranges(rec.get("affected"))
    # expand each affected block as separate range row for this vuln
    for blk in fr:
        range_rows.append({
            "id": rec.get("id"),
            "cve": row["cve"],
            "versions_count": len(blk.get("versions") or []),
            "events": blk.get("events") or []
        })

df = pd.DataFrame(norm_rows)
for col in ["published","modified"]:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True).dt.tz_localize(None)
df["severity_max"] = pd.to_numeric(df["severity_max"], errors="coerce")

df_ranges = pd.DataFrame(range_rows)
_print(f"Normalized vulns: {df.shape[0]}, range records: {df_ranges.shape[0]}")

# Save snapshots
df.to_csv(OUT_DIR / "log4j_vulns.csv", index=False)
df_ranges.to_csv(OUT_DIR / "log4j_vuln_ranges_raw.csv", index=False)


Normalized vulns: 7, range records: 16


In [ ]:

# 1) Timeline of Log4j vulnerabilities (published dates) with severity overlay
s = df.dropna(subset=["published"]).copy()
if not s.empty:
    s = s.sort_values("published")
    fig = plt.figure(figsize=(12,4))
    # scatter with severity on y (if present), else zeros
    y = s["severity_max"].fillna(0.0).values
    x = s["published"].values
    plt.scatter([str(v) for v in x], y, s=40, alpha=0.8)
    for xi, yi, label in zip([str(v) for v in x], y, s["cve"].fillna(s["id"]).values):
        # label only the biggest ones to keep it readable
        if yi >= 8.5:
            plt.text(xi, yi+0.15, label, rotation=90, fontsize=8, ha='center', va='bottom')
    plt.xticks(rotation=90)
    plt.title("Log4j vulnerabilities timeline (published) with severity")
    plt.xlabel("Published date"); plt.ylabel("Max severity (CVSS approx)")
    out = save_plot(fig, "log4j_timeline_severity")
    print("Saved:", out)


Saved: \mnt\data\osv_log4j_outputs_20250924_222526\log4j_timeline_severity.png


In [ ]:

# 2) Count by year
s = df.dropna(subset=["published"]).copy()
if not s.empty:
    s["year"] = s["published"].dt.year
    counts = s.groupby("year").size().reset_index(name="count")
    fig = plt.figure(figsize=(8,4))
    plt.bar(counts["year"].astype(str), counts["count"])
    plt.title("Log4j vulnerabilities by year")
    plt.xlabel("Year"); plt.ylabel("Count")
    out = save_plot(fig, "log4j_by_year")
    print("Saved:", out)


Saved: \mnt\data\osv_log4j_outputs_20250924_222526\log4j_by_year.png


In [ ]:

# 3) Version range chart: introduced → fixed for each vuln (where ranges exist)
# We'll extract earliest introduced and earliest fixed across all events per vuln.
def earliest(values):
    # choose the "smallest" semantic version-like string lexicographically as a fallback
    vals = [v for v in values if v]
    return min(vals) if vals else None

def earliest_fixed_event(events):
    fixed = [e.get("fixed") for e in events if e.get("fixed")]
    return earliest(fixed)

def earliest_introduced_event(events):
    intr = [e.get("introduced") for e in events if e.get("introduced")]
    return earliest(intr)

agg = []
for _, r in df_ranges.iterrows():
    evs = r["events"] or []
    agg.append({
        "id": r["id"],
        "cve": r["cve"],
        "introduced": earliest_introduced_event(evs),
        "fixed": earliest_fixed_event(evs),
    })
vf = pd.DataFrame(agg).drop_duplicates(subset=["id"])

# Save the per-vuln earliest range summary
vf.to_csv(OUT_DIR / "log4j_vuln_ranges_earliest.csv", index=False)

# Plot as a simple categorical span chart (ordered by introduced version string)
vf_plot = vf.dropna(subset=["introduced"]).copy()
if not vf_plot.empty:
    vf_plot = vf_plot.sort_values(["introduced", "fixed", "cve", "id"])
    ylabels = (vf_plot["cve"].fillna(vf_plot["id"])).tolist()
    y_pos = np.arange(len(ylabels))
    fig = plt.figure(figsize=(10, max(4, len(ylabels) * 0.4)))
    for i, row in enumerate(vf_plot.itertuples(index=False)):
        x0 = row.introduced
        x1 = row.fixed or ""  # if not fixed in events, leave open-ended
        # Represent versions as string positions along the x-axis:
        # We'll map the observed version strings to positions for a pseudo-axis.
    # Map unique version strings to x positions
    versions = pd.unique(pd.concat([vf_plot["introduced"], vf_plot["fixed"].dropna()]))
    version_to_x = {v: i for i, v in enumerate(sorted(versions))}
    for i, row in enumerate(vf_plot.itertuples(index=False)):
        x0 = version_to_x.get(row.introduced, 0)
        x1 = version_to_x.get(row.fixed, x0 + 0.5) if row.fixed else x0 + 0.5
        plt.plot([x0, x1], [i, i], linewidth=4)
    plt.yticks(y_pos, ylabels)
    plt.xticks(range(len(version_to_x)), list(version_to_x.keys()), rotation=90)
    plt.title("Log4j vulnerabilities: affected version span (introduced → fixed)")
    plt.xlabel("Version (ordinal by string)"); plt.ylabel("Vulnerability")
    plt.tight_layout()
    out = save_plot(fig, "log4j_version_spans")
    print("Saved:", out)


Saved: \mnt\data\osv_log4j_outputs_20250924_222526\log4j_version_spans.png


In [ ]:

# 4) Time-to-fix histogram: published → earliest fixed event (days)
# We need published date + earliest fixed version date. The API does not give fix dates directly;
# so we approximate using the published date and presence of a 'fixed' version.
# (If you have repo/release metadata, you could resolve release dates per version.)
s_pub = df.set_index("id")[["published"]]
vf2 = vf.set_index("id")[["fixed"]].join(s_pub, how="left").reset_index()
# keep only those with a fixed version string; we don't know when that version released,
# but we can mark how many vulns have a fixed version vs open-ended.
has_fix_rate = vf2["fixed"].notna().mean() if not vf2.empty else np.nan
pd.DataFrame({"metric": ["has_fix_version_rate"], "value": [has_fix_rate]}).to_csv(OUT_DIR / "log4j_metric_fix_version_rate.csv", index=False)
print("Has fixed-version listed (share):", has_fix_rate)

# NOTE: Without release dates, we cannot compute calendar days to fix reliably here.
# Instead, we export the ranges so you can join against Maven Central release dates if desired.


Has fixed-version listed (share): 1.0
